# ChatPromptTemplate的高级特性

## 1、部分变量预填充：partial()

举例：

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

template = ChatPromptTemplate.from_messages([
    ("system","你是{role},目标用户是{audience}"),
    ("user","{task}")
])

result1 = template.invoke({"role":"导游","audience":"游客","task":"介绍一下北京的故宫"})
result2 = template.invoke({"role":"导游","audience":"游客","task":"介绍一下北京的天安门"})

print(result1)
print(result2)

messages=[SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下北京的故宫', additional_kwargs={}, response_metadata={})]
messages=[SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下北京的天安门', additional_kwargs={}, response_metadata={})]


上述代码，可以使用partial()优化

In [4]:
from langchain_core.prompts import ChatPromptTemplate

#原始模板
template = ChatPromptTemplate.from_messages([
    ("system","你是{role},目标用户是{audience}"),
    ("user","{task}")
])

#部分变量的预填充
final_template = template.partial(role="导游",audience="游客")

result1 = final_template.invoke({"task":"给游客介绍一下天安门"})
result2 = final_template.invoke({"task":"给游客介绍一下颐和园"})\

print(result1)
print(result2)

messages=[SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='给游客介绍一下天安门', additional_kwargs={}, response_metadata={})]
messages=[SystemMessage(content='你是导游,目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='给游客介绍一下颐和园', additional_kwargs={}, response_metadata={})]


举例：为不同部门创建专用模板

In [6]:
#场景：为不同部门创建专用模板
from langchain_core.prompts import ChatPromptTemplate

#创建原始模板
template = ChatPromptTemplate.from_messages([
    ("system","你是{department},的{role}"),
    ("user","{task}")
])

# 部分变量的预填充：IT 部门
result1 = template.partial(department="IT 部门",role="技术支持")

# 部分变量的预填充：销售部门
result2 = template.partial(department="销售部门",role="销售顾问")

# 需要使用那个部门的模板就调用那个部门的预填充
result1.invoke({"task":"可以帮我解决一下这问题吗"})

ChatPromptValue(messages=[SystemMessage(content='你是IT 部门,的技术支持', additional_kwargs={}, response_metadata={}), HumanMessage(content='可以帮我解决一下这问题吗', additional_kwargs={}, response_metadata={})])

## 2、消息占位符

### 2.1 使用placeholder

举例：

In [8]:
template = ChatPromptTemplate.from_messages([
    ("system","我是一个AI助手"),
    ("placeholder","{conversation}")
])

result = template.invoke({
    "conversation":[
     ("human","明天的天气怎么样？"),
     ("ai","明天天气晴朗"),
     ("human","后天的天气怎么样？")
  ]
})

print(result)

messages=[SystemMessage(content='我是一个AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='明天的天气怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='明天天气晴朗', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='后天的天气怎么样？', additional_kwargs={}, response_metadata={})]


### 2.2使用MessagesPlaceholder

举例：

In [9]:
from langchain_core.prompts import MessagesPlaceholder

template = ChatPromptTemplate.from_messages([
    ("system","我是一个AI助手"),
    MessagesPlaceholder(variable_name="conversation")
])

result = template.invoke({
    #这里使用的是元组列表格式
    "conversation":[
        ("human","明天天气怎么样"),
        ("ai","明天天气晴朗"),
        ("human","后天的天气怎么样？")
    ]
})

print(result)

messages=[SystemMessage(content='我是一个AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='明天天气怎么样', additional_kwargs={}, response_metadata={}), AIMessage(content='明天天气晴朗', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='后天的天气怎么样？', additional_kwargs={}, response_metadata={})]


In [10]:
from langchain_core.messages import HumanMessage, AIMessage

template = ChatPromptTemplate.from_messages([
    ("system","我是一个AI助手"),
    MessagesPlaceholder(variable_name="conversation")
])

result = template.invoke({

    #使用的消息对象列表的格式
    "conversation":[
        HumanMessage("明天天气怎么样？"),
        AIMessage("明天天气清凉"),
        HumanMessage("明天天气清凉"),
    ]
})

print(result)

messages=[SystemMessage(content='我是一个AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='明天天气怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='明天天气清凉', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='明天天气清凉', additional_kwargs={}, response_metadata={})]


举例：存储历史记录

In [11]:
template = ChatPromptTemplate.from_messages([
    ("system","你是一个非常友好的AI助手"),
    MessagesPlaceholder(variable_name="conversation"),
    ("human","{question}")
])

result = template.invoke({
    "conversation":[
        ("human","5 + 8 = ?"),
        ("ai","5 + 8 = 13"),
    ],
    "question":"结果再乘于4呢？"
})

print(result)

messages=[SystemMessage(content='你是一个非常友好的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='5 + 8 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='5 + 8 = 13', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='结果再乘于4呢？', additional_kwargs={}, response_metadata={})]


### 3.可复用的模板库

定义了具体的存放模板的py文件

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
class PromptLibrary:
    """可复用的提示词模板库"""
    TRANSLATOR = ChatPromptTemplate.from_messages([
        ("system", "你是专业翻译，精通{source_lang}和{target_lang}"),
        ("user", "翻译以下文本：\n{text}")
    ])
    CODE_REVIEWER = ChatPromptTemplate.from_messages([
        ("system", "你是{language}代码审查专家，重点关注{focus}"),
        ("user", "审查代码：\n```{language}\n{code}\n```")
    ])
    SUMMARIZER = ChatPromptTemplate.from_messages([
        ("system", "你是内容摘要专家"),
        ("user", "将以下内容总结为{num}个要点：\n{content}")
    ])
    TUTOR = ChatPromptTemplate.from_messages([
        ("system", "你是{subject}导师，学生水平：{level}"),
        ("user", "{question}")
    ])

然后在其它文件中要用到那个模板就调用那个模板

In [ ]:
#比如这里就是在PromptLibrary类中调用TRANSLATOR专业翻译模板来使用
from templates import PromptLibrary
messages = PromptLibrary.TRANSLATOR.format_messages(
    source_lang="英语",
    target_lang="中文",
    text="Hello World"
)

举例2：在包里面写了多个py文件，然后每个py文件对应不同的模板(例如：有通用的模板、有和翻译相关的模板、有和编程相关的模板等)

In [ ]:
#然后在实际应用中要用那个模板就直接去调用

# templates/
# ├── __init__.py
# ├── common.py        # 通用模板
# ├── translation.py   # 翻译相关
# └── coding.py        # 编程相关
# common.py 这里举例调用的是通用模板
from langchain_core.prompts import ChatPromptTemplate
FRIENDLY_ASSISTANT = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手"),
    ("user", "{input}")
])

### 2.4模板的组合

举例：字符串组合

In [16]:
from langchain_core.prompts import ChatPromptTemplate
# 定义可复用的部分
role_part = "你是一个{domain}专家。"
style_part = "回答风格：{style}。"
constraint_part = "限制：{constraint}。"
# 组合
full_system = role_part + style_part + constraint_part
template = ChatPromptTemplate.from_messages([
    ("system", full_system),
    ("user", "{question}")
])
result= template.invoke({
         "domain":"编程",
         "style":"简介易懂",
         "constraint":"无限制",
         "question":"帮我解决一下这个关于java的问题"
})
print(result)

messages=[SystemMessage(content='你是一个编程专家。回答风格：简介易懂。限制：无限制。', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我解决一下这个关于java的问题', additional_kwargs={}, response_metadata={})]


举例2：使用+运算符

In [17]:
template1 = ChatPromptTemplate.from_messages([
    ("system", "你是助手")
])
template2 = ChatPromptTemplate.from_messages([
    ("user", "{input}")
])
# 组合（LangChain 1.0 支持）
combined = template1 + template2

result = combined.invoke({"input":"帮我看看我的邮箱中有没有新的消息"})

print(result)

messages=[SystemMessage(content='你是助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我看看我的邮箱中有没有新的消息', additional_kwargs={}, response_metadata={})]
